# Phase 0 - dev retrieval sweep + query vectors (Colab GPU)

This notebook is **the single self-contained Colab notebook**. It:

1. clones the current code from GitHub `main`,
2. downloads the 160-page dev index from the public GitHub Release
   (or reuses an existing `/content/slidevqa_index.zip` with the same hash),
3. encodes the 40 dev gold + 41 dev probe questions directly in this notebook,
4. runs the offline retrieval sweep (keyword / visual / weighted RRF with
   lexical reranking), and
5. downloads a compact results bundle.

There is no separate query-vector notebook: the query encoding cell below is
the only embedding step, and it reuses any vectors already present under
`data/query_cache_phase0`.

**Index:** the notebook downloads the 160-page dev index from a public
GitHub Release asset automatically, verifies its SHA256, and skips the
20-30 minute index build. No upload or manual step is required. If the
release download is unavailable, it falls back to building from SlideVQA.

The sweep is deterministic and does not call Gemini/Groq. It only needs the
Colab GPU for ColQwen embeddings.

Runtime: ~15 min cold (model + release download), ~30-45 min only if the
release fallback builds the index. Results are written under
`results/demo/phase0_dev_sweep/`.


In [ ]:
!nvidia-smi || true
import torch
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Select Runtime -> Change runtime type -> GPU'


In [ ]:
%pip install -q colpali-engine qdrant-client pymupdf pillow pytesseract datasets pyarrow scipy
!pip install -q -U torchao
!apt-get -qq install -y tesseract-ocr >/dev/null


In [ ]:
%cd /content
!if [ -d altVRAG/.git ]; then git -C altVRAG fetch --depth 1 origin main && git -C altVRAG reset --hard origin/main; else git clone --depth 1 https://github.com/BiprashPandey/altVVRAG.git; fi
%cd /content/altVRAG
%pip install -q -e . --no-deps
import sys
print(sys.version)
!git log -1 --oneline


In [ ]:
import json
from pathlib import Path

DEV_GOLD = [{'qid': '2012-02-20fy11roadshow-120221022442-phpapp02_95-0', 'question': 'How much is the Trading Operating Profit in 2011?', 'answer': '12.5 bn', 'acceptable_answers': [], 'doc_id': '2012-02-20fy11roadshow-120221022442-phpapp02_95', 'page': 4, 'category': 'single-hop', 'answer_in_ocr': True, 'answer_visual_only': False, 'bridged_question': False, 'acceptable_pages': [{'doc_id': '2012-02-20fy11roadshow-120221022442-phpapp02_95', 'page': 4, 'box': [1065, 563, 1180, 592], 'sentence_ids': []}]}, {'qid': '2012-02-20fy11roadshow-120221022442-phpapp02_95-1', 'question': 'How much is the Trading Operating Profit in the year Nestlé achieved the third largest Organic Growth in 10 years?', 'answer': '12.5 bn', 'acceptable_answers': [], 'doc_id': '2012-02-20fy11roadshow-120221022442-phpapp02_95', 'page': 4, 'category': 'multi-hop', 'answer_in_ocr': True, 'answer_visual_only': False, 'bridged_question': True, 'acceptable_pages': [{'doc_id': '2012-02-20fy11roadshow-120221022442-phpapp02_95', 'page': 4, 'box': [1065, 563, 1180, 592], 'sentence_ids': []}]}, {'qid': '2012-02-20fy11roadshow-120221022442-phpapp02_95-2', 'question': 'Which achieved better performance in 2011, Organic Growth or Real Internal Growth?', 'answer': 'Organic Growth', 'acceptable_answers': [], 'doc_id': '2012-02-20fy11roadshow-120221022442-phpapp02_95', 'page': 4, 'category': 'single-hop', 'answer_in_ocr': True, 'answer_visual_only': False, 'bridged_question': False, 'acceptable_pages': [{'doc_id': '2012-02-20fy11roadshow-120221022442-phpapp02_95', 'page': 4, 'box': [577, 396, 823, 431], 'sentence_ids': [4]}]}, {'qid': '2012-02-20fy11roadshow-120221022442-phpapp02_95-3', 'question': 'Which achieved better performance in the year Nestlé acquired Prometheus, Organic Growth or Real Internal Growth?', 'answer': 'Organic Growth', 'acceptable_answers': [], 'doc_id': '2012-02-20fy11roadshow-120221022442-phpapp02_95', 'page': 4, 'category': 'multi-hop', 'answer_in_ocr': True, 'answer_visual_only': False, 'bridged_question': True, 'acceptable_pages': [{'doc_id': '2012-02-20fy11roadshow-120221022442-phpapp02_95', 'page': 4, 'box': [577, 396, 823, 431], 'sentence_ids': [4]}]}, {'qid': '2012-02-20fy11roadshow-120221022442-phpapp02_95-4', 'question': 'In which year did Nestlé achieve higher Organic Growth, 2003 or 2004?', 'answer': '2003', 'acceptable_answers': [], 'doc_id': '2012-02-20fy11roadshow-120221022442-phpapp02_95', 'page': 6, 'category': 'single-hop', 'answer_in_ocr': True, 'answer_visual_only': False, 'bridged_question': False, 'acceptable_pages': [{'doc_id': '2012-02-20fy11roadshow-120221022442-phpapp02_95', 'page': 6, 'box': [975, 1006, 1042, 1029], 'sentence_ids': [12]}]}, {'qid': '2012-02-20fy11roadshow-120221022442-phpapp02_95-5', 'question': 'In which year did the company that divested Eismann in 2004 achieve higher Organic Growth, 2003 or 2004?', 'answer': '2003', 'acceptable_answers': [], 'doc_id': '2012-02-20fy11roadshow-120221022442-phpapp02_95', 'page': 6, 'category': 'multi-hop', 'answer_in_ocr': True, 'answer_visual_only': False, 'bridged_question': True, 'acceptable_pages': [{'doc_id': '2012-02-20fy11roadshow-120221022442-phpapp02_95', 'page': 6, 'box': [975, 1006, 1042, 1029], 'sentence_ids': [12]}]}, {'qid': '2012-02-20fy11roadshow-120221022442-phpapp02_95-6', 'question': 'What is the name of the brand which Nestlé acquired in 2000?', 'answer': 'PowerBar', 'acceptable_answers': [], 'doc_id': '2012-02-20fy11roadshow-120221022442-phpapp02_95', 'page': 11, 'category': 'image-only', 'answer_in_ocr': False, 'answer_visual_only': True, 'bridged_question': True, 'acceptable_pages': []}, {'qid': '2012-02-20fy11roadshow-120221022442-phpapp02_95-7', 'question': 'What is the name of the brand which Nestlé acquired in the year Findus was divested?', 'answer': 'PowerBar', 'acceptable_answers': [], 'doc_id': '2012-02-20fy11roadshow-120221022442-phpapp02_95', 'page': 11, 'category': 'image-only', 'answer_in_ocr': False, 'answer_visual_only': True, 'bridged_question': True, 'acceptable_pages': []}, {'qid': '2012-02-20fy11roadshow-120221022442-phpapp02_95-8', 'question': "What is the percentage of Nestlé's 10-year annual average of the real internal growth?", 'answer': '3.6', 'acceptable_answers': [], 'doc_id': '2012-02-20fy11roadshow-120221022442-phpapp02_95', 'page': 6, 'category': 'single-hop', 'answer_in_ocr': True, 'answer_visual_only': False, 'bridged_question': False, 'acceptable_pages': [{'doc_id': '2012-02-20fy11roadshow-120221022442-phpapp02_95', 'page': 6, 'box': [481, 581, 563, 608], 'sentence_ids': [8]}]}, {'qid': '2012-02-20fy11roadshow-120221022442-phpapp02_95-9', 'question': 'What is the percentage of the 10-year annual average of the Real Internal Growth for the company which has the tagline "Good Food, Good Life"?', 'answer': '3.6', 'acceptable_answers': [], 'doc_id': '2012-02-20fy11roadshow-120221022442-phpapp02_95', 'page': 0, 'category': 'image-only', 'answer_in_ocr': False, 'answer_visual_only': True, 'bridged_question': True, 'acceptable_pages': [{'doc_id': '2012-02-20fy11roadshow-120221022442-phpapp02_95', 'page': 4, 'box': [1063, 317, 1132, 344], 'sentence_ids': [24]}]}, {'qid': '0903organizingforbiandbigdatainthe21stcentury-clean-140922112024-phpapp02_95-10', 'question': 'How many kinds of BICC Key Skills are there?', 'answer': '3', 'acceptable_answers': [], 'doc_id': '0903organizingforbiandbigdatainthe21stcentury-clean-140922112024-phpapp02_95', 'page': 7, 'category': 'image-only', 'answer_in_ocr': False, 'answer_visual_only': True, 'bridged_question': False, 'acceptable_pages': [{'doc_id': '0903organizingforbiandbigdatainthe21stcentury-clean-140922112024-phpapp02_95', 'page': 0, 'box': [1684, 1110, 1838, 1165], 'sentence_ids': [2]}]}, {'qid': '0903organizingforbiandbigdatainthe21stcentury-clean-140922112024-phpapp02_95-11', 'question': 'How many kinds of Business Intelligence Competence Center Key Skills are there?', 'answer': '3', 'acceptable_answers': [], 'doc_id': '0903organizingforbiandbigdatainthe21stcentury-clean-140922112024-phpapp02_95', 'page': 3, 'category': 'image-only', 'answer_in_ocr': False, 'answer_visual_only': True, 'bridged_question': False, 'acceptable_pages': [{'doc_id': '0903organizingforbiandbigdatainthe21stcentury-clean-140922112024-phpapp02_95', 'page': 0, 'box': [1684, 1110, 1838, 1165], 'sentence_ids': [2]}]}, {'qid': '0903organizingforbiandbigdatainthe21stcentury-clean-140922112024-phpapp02_95-12', 'question': 'What is the total percentage that wait Minutes and Hours for an average BI request?', 'answer': '65', 'acceptable_answers': [], 'doc_id': '0903organizingforbiandbigdatainthe21stcentury-clean-140922112024-phpapp02_95', 'page': 15, 'category': 'image-only', 'answer_in_ocr': False, 'answer_visual_only': True, 'bridged_question': False, 'acceptable_pages': []}, {'qid': '0903organizingforbiandbigdatainthe21stcentury-clean-140922112024-phpapp02_95-13', 'question': 'What is the biggest reason why users sometimes choose not to use enterprise BI applications?', 'answer': 'Do not have all the data you need', 'acceptable_answers': [], 'doc_id': '0903organizingforbiandbigdatainthe21stcentury-clean-140922112024-phpapp02_95', 'page': 16, 'category': 'single-hop', 'answer_in_ocr': True, 'answer_visual_only': False, 'bridged_question': False, 'acceptable_pages': [{'doc_id': '0903organizingforbiandbigdatainthe21stcentury-clean-140922112024-phpapp02_95', 'page': 16, 'box': [248, 625, 838, 660], 'sentence_ids': [5]}]}, {'qid': '0903organizingforbiandbigdatainthe21stcentury-clean-140922112024-phpapp02_95-14', 'question': 'What is the biggest reason why users sometimes choose not to use the applications, data of which are not mostly trusted by 8% users?', 'answer': 'Do not have all the data you need', 'acceptable_answers': [], 'doc_id': '0903organizingforbiandbigdatainthe21stcentury-clean-140922112024-phpapp02_95', 'page': 16, 'category': 'multi-hop', 'answer_in_ocr': True, 'answer_visual_only': False, 'bridged_question': True, 'acceptable_pages': [{'doc_id': '0903organizingforbiandbigdatainthe21stcentury-clean-140922112024-phpapp02_95', 'page': 16, 'box': [248, 625, 838, 660], 'sentence_ids': [5]}]}, {'qid': '0903organizingforbiandbigdatainthe21stcentury-clean-140922112024-phpapp02_95-15', 'question': 'Which receives lower percentage of answers, Enterprise applications "Are too complex, complicated, cumbersome to use" or "Take too long to get results"?', 'answer': 'Take too long to get results', 'acceptable_answers': [], 'doc_id': '0903organizingforbiandbigdatainthe21stcentury-clean-140922112024-phpapp02_95', 'page': 16, 'category': 'single-hop', 'answer_in_ocr': True, 'answer_visual_only': False, 'bridged_question': False, 'acceptable_pages': [{'doc_id': '0903organizingforbiandbigdatainthe21stcentury-clean-140922112024-phpapp02_95', 'page': 16, 'box': [363, 1048, 838, 1085], 'sentence_ids': [11]}]}, {'qid': '0903organizingforbiandbigdatainthe21stcentury-clean-140922112024-phpapp02_95-16', 'question': 'What is the percentage of the respondents who answer "I use only enterprise BI applications"?', 'answer': '15', 'acceptable_answers': [], 'doc_id': '0903organizingforbiandbigdatainthe21stcentury-clean-140922112024-phpapp02_95', 'page': 18, 'category': 'single-hop', 'answer_in_ocr': True, 'answer_visual_only': False, 'bridged_question': False, 'acceptable_pages': [{'doc_id': '0903organizingforbiandbigdatainthe21stcentury-clean-140922112024-phpapp02_95', 'page': 18, 'box': [1430, 656, 1492, 685], 'sentence_ids': [7]}]}, {'qid': 'alcatel-lucenteesoverview-120403090342-phpapp01_95-17', 'question': 'What is the title of the presentation that ALCATEL-LUCENT made for the OVERVIEW of 2012?', 'answer': 'AT THE SPEED OF IDEAS', 'acceptable_answers': ['ALCATEL-LUCENT OVERVIEW 2012', 'Alcatel-Lucent Overview 2012'], 'doc_id': 'alcatel-lucenteesoverview-120403090342-phpapp01_95', 'page': 0, 'category': 'image-only', 'answer_in_ocr': False, 'answer_visual_only': True, 'bridged_question': False, 'acceptable_pages': [{'doc_id': 'alcatel-lucenteesoverview-120403090342-phpapp01_95', 'page': 1, 'box': [823, 386, 1613, 438], 'sentence_ids': [2, 7]}]}, {'qid': 'alcatel-lucenteesoverview-120403090342-phpapp01_95-18', 'question': 'What is the title of the presentation that the company which had won 7 Nobel Prizes made for the OVERVIEW of 2012?', 'answer': 'AT THE SPEED OF IDEAS', 'acceptable_answers': ['ALCATEL-LUCENT OVERVIEW 2012', 'Alcatel-Lucent Overview 2012'], 'doc_id': 'alcatel-lucenteesoverview-120403090342-phpapp01_95', 'page': 0, 'category': 'image-only', 'answer_in_ocr': False, 'answer_visual_only': True, 'bridged_question': True, 'acceptable_pages': [{'doc_id': 'alcatel-lucenteesoverview-120403090342-phpapp01_95', 'page': 1, 'box': [823, 386, 1613, 438], 'sentence_ids': [2, 7]}]}, {'qid': 'alcatel-lucenteesoverview-120403090342-phpapp01_95-19', 'question': 'Which region of the world accounts for the highest percentage of 2010 revenues?', 'answer': 'N. America', 'acceptable_answers': [], 'doc_id': 'alcatel-lucenteesoverview-120403090342-phpapp01_95', 'page': 6, 'category': 'single-hop', 'answer_in_ocr': True, 'answer_visual_only': False, 'bridged_question': False, 'acceptable_pages': [{'doc_id': 'alcatel-lucenteesoverview-120403090342-phpapp01_95', 'page': 6, 'box': [296, 267, 469, 290], 'sentence_ids': [2]}]}, {'qid': 'alcatel-lucenteesoverview-120403090342-phpapp01_95-20', 'question': 'Which region of the world accounts for the highest percentage of revenues in the year 12% GROWTH is achieved?', 'answer': 'N. America', 'acceptable_answers': [], 'doc_id': 'alcatel-lucenteesoverview-120403090342-phpapp01_95', 'page': 6, 'category': 'multi-hop', 'answer_in_ocr': True, 'answer_visual_only': False, 'bridged_question': True, 'acceptable_pages': [{'doc_id': 'alcatel-lucenteesoverview-120403090342-phpapp01_95', 'page': 6, 'box': [296, 267, 469, 290], 'sentence_ids': [2]}]}, {'qid': 'alcatel-lucenteesoverview-120403090342-phpapp01_95-21', 'question': 'Where does Alcatel-Lucent have the IP Transformation Center in Europe?', 'answer': 'Belgium', 'acceptable_answers': [], 'doc_id': 'alcatel-lucenteesoverview-120403090342-phpapp01_95', 'page': 6, 'category': 'single-hop', 'answer_in_ocr': True, 'answer_visual_only': False, 'bridged_question': False, 'acceptable_pages': [{'doc_id': 'alcatel-lucenteesoverview-120403090342-phpapp01_95', 'page': 6, 'box': [1001, 561, 1109, 606], 'sentence_ids': [10]}]}, {'qid': 'alcatel-lucenteesoverview-120403090342-phpapp01_95-22', 'question': 'Where does the company which was awarded 2400 Patents in 2010 have the IP Transformation Center in Europe?', 'answer': 'Belgium', 'acceptable_answers': [], 'doc_id': 'alcatel-lucenteesoverview-120403090342-phpapp01_95', 'page': 4, 'category': 'image-only', 'answer_in_ocr': False, 'answer_visual_only': True, 'bridged_question': True, 'acceptable_pages': [{'doc_id': 'alcatel-lucenteesoverview-120403090342-phpapp01_95', 'page': 5, 'box': [311, 509, 431, 544], 'sentence_ids': [4]}]}, {'qid': 'alcatel-lucenteesoverview-120403090342-phpapp01_95-23', 'question': 'How many Mobile Operators are listed as References / proof of our expertise?', 'answer': '12', 'acceptable_answers': [], 'doc_id': 'alcatel-lucenteesoverview-120403090342-phpapp01_95', 'page': 17, 'category': 'image-only', 'answer_in_ocr': False, 'answer_visual_only': True, 'bridged_question': False, 'acceptable_pages': [{'doc_id': 'alcatel-lucenteesoverview-120403090342-phpapp01_95', 'page': 0, 'box': [252, 1080, 402, 1128], 'sentence_ids': [0]}]}, {'qid': '8eeomnwqsmdmin6levsv-signature-2693c633d5e2c6d5cfd1372a7aa1d1a8df13f59c942a848ca9497686d609bddd-poli-150601151510-lva1-app6892_95-24', 'question': 'Which Activity is undertaken by more smartphone users, Email services or Social networks?', 'answer': 'Social networks', 'acceptable_answers': [], 'doc_id': '8eeomnwqsmdmin6levsv-signature-2693c633d5e2c6d5cfd1372a7aa1d1a8df13f59c942a848ca9497686d609bddd-poli-150601151510-lva1-a', 'page': 4, 'category': 'single-hop', 'answer_in_ocr': True, 'answer_visual_only': False, 'bridged_question': False, 'acceptable_pages': [{'doc_id': '8eeomnwqsmdmin6levsv-signature-2693c633d5e2c6d5cfd1372a7aa1d1a8df13f59c942a848ca9497686d609bddd-poli-150601151510-lva1-a', 'page': 4, 'box': [77, 567, 246, 583], 'sentence_ids': [3]}]}, {'qid': '8eeomnwqsmdmin6levsv-signature-2693c633d5e2c6d5cfd1372a7aa1d1a8df13f59c942a848ca9497686d609bddd-poli-150601151510-lva1-app6892_95-25', 'question': 'Which country spend the most in Mobile display ads in 2013, according to IAB Europe AdEx Benchmark (2013)?', 'answer': 'Turkey', 'acceptable_answers': [], 'doc_id': '8eeomnwqsmdmin6levsv-signature-2693c633d5e2c6d5cfd1372a7aa1d1a8df13f59c942a848ca9497686d609bddd-poli-150601151510-lva1-a', 'page': 5, 'category': 'image-only', 'answer_in_ocr': False, 'answer_visual_only': True, 'bridged_question': False, 'acceptable_pages': []}, {'qid': '8eeomnwqsmdmin6levsv-signature-2693c633d5e2c6d5cfd1372a7aa1d1a8df13f59c942a848ca9497686d609bddd-poli-150601151510-lva1-app6892_95-26', 'question': 'Which country spend the most in Mobile display ads in the year 79300 Romanian use Twitter, according to IAB Europe AdEx Benchmark (2013)?', 'answer': 'Turkey', 'acceptable_answers': [], 'doc_id': '8eeomnwqsmdmin6levsv-signature-2693c633d5e2c6d5cfd1372a7aa1d1a8df13f59c942a848ca9497686d609bddd-poli-150601151510-lva1-a', 'page': 5, 'category': 'image-only', 'answer_in_ocr': False, 'answer_visual_only': True, 'bridged_question': True, 'acceptable_pages': []}, {'qid': '8eeomnwqsmdmin6levsv-signature-2693c633d5e2c6d5cfd1372a7aa1d1a8df13f59c942a848ca9497686d609bddd-poli-150601151510-lva1-app6892_95-27', 'question': 'What is the total percentage of 4G users with the 1 - 500 MB plan and the 501 MB - 1 GB plan in the UK?', 'answer': '53', 'acceptable_answers': [], 'doc_id': '8eeomnwqsmdmin6levsv-signature-2693c633d5e2c6d5cfd1372a7aa1d1a8df13f59c942a848ca9497686d609bddd-poli-150601151510-lva1-a', 'page': 6, 'category': 'image-only', 'answer_in_ocr': False, 'answer_visual_only': True, 'bridged_question': False, 'acceptable_pages': [{'doc_id': '8eeomnwqsmdmin6levsv-signature-2693c633d5e2c6d5cfd1372a7aa1d1a8df13f59c942a848ca9497686d609bddd-poli-150601151510-lva1-a', 'page': 5, 'box': [998, 740, 1094, 763], 'sentence_ids': [11]}]}, {'qid': '8eeomnwqsmdmin6levsv-signature-2693c633d5e2c6d5cfd1372a7aa1d1a8df13f59c942a848ca9497686d609bddd-poli-150601151510-lva1-app6892_95-28', 'question': 'Which country had the higher percentage of Tablet Pageviews in June 2014, Russia or Hungary?', 'answer': 'Russia', 'acceptable_answers': [], 'doc_id': '8eeomnwqsmdmin6levsv-signature-2693c633d5e2c6d5cfd1372a7aa1d1a8df13f59c942a848ca9497686d609bddd-poli-150601151510-lva1-a', 'page': 8, 'category': 'image-only', 'answer_in_ocr': False, 'answer_visual_only': True, 'bridged_question': False, 'acceptable_pages': []}, {'qid': '8eeomnwqsmdmin6levsv-signature-2693c633d5e2c6d5cfd1372a7aa1d1a8df13f59c942a848ca9497686d609bddd-poli-150601151510-lva1-app6892_95-29', 'question': 'Which country had the higher percentage of Tablet Pageviews in June 2014, Russia or the country where 61 million euros was spent in online display ads in 2013?', 'answer': 'Russia', 'acceptable_answers': [], 'doc_id': '8eeomnwqsmdmin6levsv-signature-2693c633d5e2c6d5cfd1372a7aa1d1a8df13f59c942a848ca9497686d609bddd-poli-150601151510-lva1-a', 'page': 5, 'category': 'image-only', 'answer_in_ocr': False, 'answer_visual_only': True, 'bridged_question': True, 'acceptable_pages': []}, {'qid': '8eeomnwqsmdmin6levsv-signature-2693c633d5e2c6d5cfd1372a7aa1d1a8df13f59c942a848ca9497686d609bddd-poli-150601151510-lva1-app6892_95-30', 'question': 'Which OS was used more among mobile Facebook users in October 2014, Android or Windows Mobile?', 'answer': 'Android', 'acceptable_answers': [], 'doc_id': '8eeomnwqsmdmin6levsv-signature-2693c633d5e2c6d5cfd1372a7aa1d1a8df13f59c942a848ca9497686d609bddd-poli-150601151510-lva1-a', 'page': 9, 'category': 'image-only', 'answer_in_ocr': False, 'answer_visual_only': True, 'bridged_question': False, 'acceptable_pages': [{'doc_id': '8eeomnwqsmdmin6levsv-signature-2693c633d5e2c6d5cfd1372a7aa1d1a8df13f59c942a848ca9497686d609bddd-poli-150601151510-lva1-a', 'page': 7, 'box': [973, 398, 1046, 413], 'sentence_ids': [2]}]}, {'qid': 'a-151012171903-lva1-app6891_95-31', 'question': 'How many points are listed for TAM Notes?', 'answer': '4', 'acceptable_answers': [], 'doc_id': 'a-151012171903-lva1-app6891_95', 'page': 11, 'category': 'single-hop', 'answer_in_ocr': True, 'answer_visual_only': False, 'bridged_question': False, 'acceptable_pages': [{'doc_id': 'a-151012171903-lva1-app6891_95', 'page': 11, 'box': [413, 388, 465, 413], 'sentence_ids': [4, 18]}]}, {'qid': 'a-151012171903-lva1-app6891_95-32', 'question': 'What is the abbreviation for Sales Qualified Leads?', 'answer': 'SQL', 'acceptable_answers': [], 'doc_id': 'a-151012171903-lva1-app6891_95', 'page': 16, 'category': 'image-only', 'answer_in_ocr': False, 'answer_visual_only': True, 'bridged_question': False, 'acceptable_pages': []}, {'qid': 'a-151012171903-lva1-app6891_95-33', 'question': 'How many Partner companies are listed in the example of Go to Market Notes?', 'answer': '3', 'acceptable_answers': [], 'doc_id': 'a-151012171903-lva1-app6891_95', 'page': 17, 'category': 'image-only', 'answer_in_ocr': False, 'answer_visual_only': True, 'bridged_question': False, 'acceptable_pages': [{'doc_id': 'a-151012171903-lva1-app6891_95', 'page': 1, 'box': [409, 531, 448, 569], 'sentence_ids': [3]}]}, {'qid': 'advertisingpsychologyfinal-090705155755-phpapp02_95-34', 'question': 'What is the Major objective other than capturing attention and making a useful lasting impression?', 'answer': 'Arouse and hold interest', 'acceptable_answers': [], 'doc_id': 'advertisingpsychologyfinal-090705155755-phpapp02_95', 'page': 7, 'category': 'image-only', 'answer_in_ocr': False, 'answer_visual_only': True, 'bridged_question': False, 'acceptable_pages': []}, {'qid': 'advertisingpsychologyfinal-090705155755-phpapp02_95-35', 'question': 'What are listed as the examples of Social characteristics in A framework of psychological meaning?', 'answer': 'gender, social class, marital status, occupation', 'acceptable_answers': [], 'doc_id': 'advertisingpsychologyfinal-090705155755-phpapp02_95', 'page': 16, 'category': 'single-hop', 'answer_in_ocr': True, 'answer_visual_only': False, 'bridged_question': False, 'acceptable_pages': [{'doc_id': 'advertisingpsychologyfinal-090705155755-phpapp02_95', 'page': 16, 'box': [1734, 892, 1928, 1050], 'sentence_ids': [29]}]}, {'qid': 'a-3-c2b8b6654f47bdfbff1433386996d789694f921a-150414014636-conversion-gate01_95-36', 'question': 'What is a term that refers to the way work is assigned to the team?', 'answer': 'Command', 'acceptable_answers': [], 'doc_id': 'a-3-c2b8b6654f47bdfbff1433386996d789694f921a-150414014636-conversion-gate01_95', 'page': 15, 'category': 'image-only', 'answer_in_ocr': False, 'answer_visual_only': True, 'bridged_question': False, 'acceptable_pages': []}, {'qid': '1shangxichinassubnationaldebts-issuesandsuggestions-160229201053_95-37', 'question': 'In which year is the percentage of the Subnational Debts guaranteed by Governments higher, 2015 or 2016?', 'answer': '2015', 'acceptable_answers': [], 'doc_id': '1shangxichinassubnationaldebts-issuesandsuggestions-160229201053_95', 'page': 9, 'category': 'single-hop', 'answer_in_ocr': True, 'answer_visual_only': False, 'bridged_question': False, 'acceptable_pages': [{'doc_id': '1shangxichinassubnationaldebts-issuesandsuggestions-160229201053_95', 'page': 9, 'box': [1874, 413, 1965, 446], 'sentence_ids': [1, 4, 6, 17]}]}, {'qid': '1shangxichinassubnationaldebts-issuesandsuggestions-160229201053_95-38', 'question': 'In which year may Subnational Debts be relieved more by Government, 2015 or 2016?', 'answer': '2015', 'acceptable_answers': [], 'doc_id': '1shangxichinassubnationaldebts-issuesandsuggestions-160229201053_95', 'page': 9, 'category': 'single-hop', 'answer_in_ocr': True, 'answer_visual_only': False, 'bridged_question': False, 'acceptable_pages': [{'doc_id': '1shangxichinassubnationaldebts-issuesandsuggestions-160229201053_95', 'page': 9, 'box': [1874, 413, 1965, 446], 'sentence_ids': [1, 4, 6, 17]}]}, {'qid': '1shangxichinassubnationaldebts-issuesandsuggestions-160229201053_95-39', 'question': 'Which are the bigger Debts to be repaid as of June 2013, Provincial-level government debts or Municipal-level government debts?', 'answer': 'Municipal-level government debts', 'acceptable_answers': [], 'doc_id': '1shangxichinassubnationaldebts-issuesandsuggestions-160229201053_95', 'page': 5, 'category': 'single-hop', 'answer_in_ocr': True, 'answer_visual_only': False, 'bridged_question': False, 'acceptable_pages': [{'doc_id': '1shangxichinassubnationaldebts-issuesandsuggestions-160229201053_95', 'page': 5, 'box': [744, 504, 1024, 531], 'sentence_ids': []}]}]
DEV_PROBES = [{'qid': 'unans-00', 'question': 'What was the closing stock price of Alcatel-Lucent on December 31, 2013?', 'answer': 'UNANSWERABLE', 'category': 'unanswerable', 'probe_category': 'absent-attribute', 'why': 'Run 7 verified probe (re-categorized)'}, {'qid': 'unans-01', 'question': 'How many kinds of BICC Key Skills are there in 2020?', 'answer': 'UNANSWERABLE', 'category': 'unanswerable', 'probe_category': 'absent-attribute', 'why': 'attribute question scoped to an absent year'}, {'qid': 'unans-02', 'question': "What is the percentage of Nestle's 10-year annual average of the real internal growth as reported in 2021?", 'answer': 'UNANSWERABLE', 'category': 'unanswerable', 'probe_category': 'absent-attribute', 'why': 'shadowed by the present 10-year average (q8)'}, {'qid': 'unans-03', 'question': 'Which are the bigger Debts to be repaid as of March 2019, Provincial-level government debts or Municipal-level government debts?', 'answer': 'UNANSWERABLE', 'category': 'unanswerable', 'probe_category': 'absent-attribute', 'why': 'debt table shadow with absent date'}, {'qid': 'unans-04', 'question': 'How much is the Trading Operating Profit in 2019?', 'answer': 'UNANSWERABLE', 'category': 'unanswerable', 'probe_category': 'absent-attribute', 'why': 'qid 2_95-0: years swapped to absent [2019]'}, {'qid': 'unans-05', 'question': 'Which achieved better performance in 2020, Organic Growth or Real Internal Growth?', 'answer': 'UNANSWERABLE', 'category': 'unanswerable', 'probe_category': 'absent-attribute', 'why': 'qid 2_95-2: years swapped to absent [2020]'}, {'qid': 'unans-06', 'question': 'In which year did Nestlé achieve higher Organic Growth, 2021 or 2022?', 'answer': 'UNANSWERABLE', 'category': 'unanswerable', 'probe_category': 'absent-attribute', 'why': 'qid 2_95-4: years swapped to absent [2021, 2022]'}, {'qid': 'unans-07', 'question': 'Which region of the world accounts for the highest percentage of 2023 revenues?', 'answer': 'UNANSWERABLE', 'category': 'unanswerable', 'probe_category': 'absent-attribute', 'why': 'qid _95-19: years swapped to absent [2023]'}, {'qid': 'unans-08', 'question': 'In which year is the percentage of the Subnational Debts guaranteed by Governments higher, 2025 or 1999?', 'answer': 'UNANSWERABLE', 'category': 'unanswerable', 'probe_category': 'absent-attribute', 'why': 'qid _95-37: years swapped to absent [2025, 1999]'}, {'qid': 'unans-09', 'question': 'In which year may Subnational Debts be relieved more by Government, 1998 or 1997?', 'answer': 'UNANSWERABLE', 'category': 'unanswerable', 'probe_category': 'absent-attribute', 'why': 'qid _95-38: years swapped to absent [1998, 1997]'}, {'qid': 'unans-10', 'question': 'Which are the bigger Debts to be repaid as of June 1995, Provincial-level government debts or Municipal-level government debts?', 'answer': 'UNANSWERABLE', 'category': 'unanswerable', 'probe_category': 'absent-attribute', 'why': 'qid _95-39: years swapped to absent [1995]'}, {'qid': 'unans-11', 'question': 'Which are the bigger Debts to be repaid, Provincial-level government debts or Municipal-level government debts in June 2012?', 'answer': 'UNANSWERABLE', 'category': 'unanswerable', 'probe_category': 'near-miss', 'why': 'shadows the June 2013 debt table (q39); June 2012 absent'}, {'qid': 'unans-12', 'question': 'Which country had the higher percentage of Tablet Pageviews in July 2014, Russia or Hungary?', 'answer': 'UNANSWERABLE', 'category': 'unanswerable', 'probe_category': 'near-miss', 'why': 'shadows the June 2014 tablet table (q28)'}, {'qid': 'unans-13', 'question': "What is the percentage of Nestle's 20-year annual average of the real internal growth?", 'answer': 'UNANSWERABLE', 'category': 'unanswerable', 'probe_category': 'near-miss', 'why': 'shadows the 10-year average (q8)'}, {'qid': 'unans-14', 'question': 'What is the total percentage that wait Minutes and Hours for an average BI request in 2020?', 'answer': 'UNANSWERABLE', 'category': 'unanswerable', 'probe_category': 'near-miss', 'why': 'shadows the BI on-demand waiting poll (q12)'}, {'qid': 'unans-15', 'question': 'Which Activity is undertaken by more laptop users, Email services or Social networks?', 'answer': 'UNANSWERABLE', 'category': 'unanswerable', 'probe_category': 'near-miss', 'why': 'shadows the smartphone activity table (q24)'}, {'qid': 'unans-16', 'question': 'Which OS was used more among mobile Facebook users in November 2014, Android or Windows Mobile?', 'answer': 'UNANSWERABLE', 'category': 'unanswerable', 'probe_category': 'near-miss', 'why': 'shadows the October 2014 OS table (q30)'}, {'qid': 'unans-17', 'question': 'Which country spend the most in Mobile display ads in 2019, according to IAB Europe AdEx Benchmark?', 'answer': 'UNANSWERABLE', 'category': 'unanswerable', 'probe_category': 'near-miss', 'why': 'shadows the 2013 ad-spend chart (q25/26)'}, {'qid': 'unans-18', 'question': 'Which region of the world accounts for the highest percentage of 2019 revenues?', 'answer': 'UNANSWERABLE', 'category': 'unanswerable', 'probe_category': 'near-miss', 'why': 'shadows the 2010 revenue mix (q19)'}, {'qid': 'unans-19', 'question': 'How many employees does Zephyrix Analytics have in Zurich?', 'answer': 'UNANSWERABLE', 'category': 'unanswerable', 'probe_category': 'entity-swap', 'why': 'Run 7 verified probe (re-categorized)'}, {'qid': 'unans-20', 'question': 'What was the total revenue of Zynga in 2011?', 'answer': 'UNANSWERABLE', 'category': 'unanswerable', 'probe_category': 'entity-swap', 'why': 'Run 7 verified probe (re-categorized)'}, {'qid': 'unans-21', 'question': 'How many 5G subscriptions were active worldwide in 2020?', 'answer': 'UNANSWERABLE', 'category': 'unanswerable', 'probe_category': 'entity-swap', 'why': 'Run 7 verified probe (re-categorized)'}, {'qid': 'unans-22', 'question': 'How much did Nestle pay to acquire Wyeth Nutrition?', 'answer': 'UNANSWERABLE', 'category': 'unanswerable', 'probe_category': 'entity-swap', 'why': 'Run 7 verified probe (re-categorized)'}, {'qid': 'unans-23', 'question': "What was Siemens' organic growth in 2011 according to the Nestle roadshow?", 'answer': 'UNANSWERABLE', 'category': 'unanswerable', 'probe_category': 'entity-swap', 'why': 'Run 7 verified probe (re-categorized)'}, {'qid': 'unans-24', 'question': 'Which company acquired Alcatel-Lucent in 2016?', 'answer': 'UNANSWERABLE', 'category': 'unanswerable', 'probe_category': 'entity-swap', 'why': 'Run 7 verified probe (re-categorized)'}, {'qid': 'unans-25', 'question': 'What was the market share of Xiaomi in the global smartphone market?', 'answer': 'UNANSWERABLE', 'category': 'unanswerable', 'probe_category': 'entity-swap', 'why': 'Xiaomi never appears'}, {'qid': 'unans-26', 'question': 'How many employees does Samsung Electronics have in Seoul?', 'answer': 'UNANSWERABLE', 'category': 'unanswerable', 'probe_category': 'entity-swap', 'why': 'Samsung never appears'}, {'qid': 'unans-27', 'question': 'How many users does WeChat have in Europe?', 'answer': 'UNANSWERABLE', 'category': 'unanswerable', 'probe_category': 'entity-swap', 'why': 'WeChat never appears'}, {'qid': 'unans-28', 'question': 'How many employees does Microsoft have worldwide?', 'answer': 'UNANSWERABLE', 'category': 'unanswerable', 'probe_category': 'entity-swap', 'why': 'Microsoft never appears'}, {'qid': 'unans-29', 'question': "What was Sony's revenue in 2011?", 'answer': 'UNANSWERABLE', 'category': 'unanswerable', 'probe_category': 'entity-swap', 'why': 'Sony never appears'}, {'qid': 'unans-30', 'question': 'How many members does LinkedIn have?', 'answer': 'UNANSWERABLE', 'category': 'unanswerable', 'probe_category': 'entity-swap', 'why': 'LinkedIn never appears'}, {'qid': 'unans-31', 'question': 'What is the estimated cost of the Qatar 2022 World Cup?', 'answer': 'UNANSWERABLE', 'category': 'unanswerable', 'probe_category': 'entity-swap', 'why': 'Qatar never appears'}, {'qid': 'unans-32', 'question': 'Who was the CEO of Intel who replaced Paul Otellini in 2013?', 'answer': 'UNANSWERABLE', 'category': 'unanswerable', 'probe_category': 'false-bridge', 'why': 'Run 7 verified probe (re-categorized)'}, {'qid': 'unans-33', 'question': 'What is the second largest mobile operator by revenue mentioned in the Alcatel-Lucent references?', 'answer': 'UNANSWERABLE', 'category': 'unanswerable', 'probe_category': 'false-bridge', 'why': 'Run 7 verified probe (re-categorized)'}, {'qid': 'unans-34', 'question': 'In the year Nestle was founded, what was its organic growth?', 'answer': 'UNANSWERABLE', 'category': 'unanswerable', 'probe_category': 'false-bridge', 'why': 'founding year absent; qualifier cannot resolve'}, {'qid': 'unans-35', 'question': 'What is the name of the brand which Nestle acquired in the year the Berlin Wall fell?', 'answer': 'UNANSWERABLE', 'category': 'unanswerable', 'probe_category': 'false-bridge', 'why': 'event absent from every page'}, {'qid': 'unans-36', 'question': 'What was the revenue of the company that acquired WhatsApp?', 'answer': 'UNANSWERABLE', 'category': 'unanswerable', 'probe_category': 'false-bridge', 'why': 'acquirer absent'}, {'qid': 'unans-37', 'question': 'Who was the CEO of Nestle when the iPhone was released?', 'answer': 'UNANSWERABLE', 'category': 'unanswerable', 'probe_category': 'false-bridge', 'why': 'temporal anchor absent'}, {'qid': 'unans-38', 'question': 'What does the Alcatel-Lucent deck say will happen after Brexit?', 'answer': 'UNANSWERABLE', 'category': 'unanswerable', 'probe_category': 'false-bridge', 'why': 'event absent'}, {'qid': 'unans-39', 'question': 'Which of these documents mentions the Amazon rainforest?', 'answer': 'UNANSWERABLE', 'category': 'unanswerable', 'probe_category': 'false-bridge', 'why': 'topic absent'}, {'qid': 'unans-40', 'question': 'What was the Alcatel-Lucent stock price on the day Twitter went public?', 'answer': 'UNANSWERABLE', 'category': 'unanswerable', 'probe_category': 'false-bridge', 'why': 'anchor event absent'}]
DEV_DECKS = ['0903organizingforbiandbigdatainthe21stcentury-clean-140922112024-phpapp02_95', '1shangxichinassubnationaldebts-issuesandsuggestions-160229201053_95', '2012-02-20fy11roadshow-120221022442-phpapp02_95', '8eeomnwqsmdmin6levsv-signature-2693c633d5e2c6d5cfd1372a7aa1d1a8df13f59c942a848ca9497686d609bddd-poli-150601151510-lva1-a', 'a-151012171903-lva1-app6891_95', 'a-3-c2b8b6654f47bdfbff1433386996d789694f921a-150414014636-conversion-gate01_95', 'advertisingpsychologyfinal-090705155755-phpapp02_95', 'alcatel-lucenteesoverview-120403090342-phpapp01_95']
EXPECTED_PAGES = 160
EXPECTED_QUESTIONS = 81

Path('data').mkdir(exist_ok=True)
Path('data/phase0_dev_gold.json').write_text(json.dumps(DEV_GOLD, indent=1), encoding='utf-8')
Path('data/phase0_dev_probes.json').write_text(json.dumps(DEV_PROBES, indent=1), encoding='utf-8')
Path('data/phase0_dev_decks.json').write_text(json.dumps(DEV_DECKS, indent=1), encoding='utf-8')
Path('data/phase0_dev_questions.json').write_text(json.dumps(
    [{'question': x['question']} for x in DEV_GOLD + DEV_PROBES], indent=1), encoding='utf-8')
print('gold', len(DEV_GOLD), 'probes', len(DEV_PROBES), 'decks', len(DEV_DECKS))
print('questions', len(DEV_GOLD) + len(DEV_PROBES))


In [ ]:
import hashlib
import json
import shutil
import sqlite3
import subprocess
import sys
import urllib.request
import zipfile
from pathlib import Path

INDEX = Path('data/index_phase0')
UPLOAD = Path('/content/slidevqa_index.zip')
INDEX_ZIP_URL = 'https://github.com/BiprashPandey/altVVRAG/releases/download/phase0-dev-index-v1/slidevqa_index.zip'
INDEX_SHA256 = 'd30ad0de69d8a1cd8de6a7a22abc0906d4a866d06b37334afad37c34d5cfa226'
FRESH = False

def sha256(path: Path) -> str:
    h = hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

def download_index() -> None:
    tmp = UPLOAD.with_suffix('.part')
    tmp.unlink(missing_ok=True)
    print('downloading dev index from GitHub Release ...')
    req = urllib.request.Request(INDEX_ZIP_URL, headers={'User-Agent': 'altVRAG-colab'})
    with urllib.request.urlopen(req, timeout=120) as r, tmp.open('wb') as f:
        total = int(r.headers.get('Content-Length') or 0)
        done = 0
        while True:
            chunk = r.read(1 << 20)
            if not chunk:
                break
            f.write(chunk)
            done += len(chunk)
            if total and done % (20 << 20) < (1 << 20):
                print(f'  {done/1e6:.0f}/{total/1e6:.0f} MB')
    got = sha256(tmp)
    if got != INDEX_SHA256:
        tmp.unlink(missing_ok=True)
        raise RuntimeError(f'index SHA256 mismatch: {got} != {INDEX_SHA256}')
    tmp.replace(UPLOAD)
    print('downloaded', UPLOAD, UPLOAD.stat().st_size / 1e6, 'MB; sha256 ok')

if UPLOAD.exists():
    if sha256(UPLOAD) == INDEX_SHA256:
        print('using cached index zip; sha256 ok')
    else:
        print('cached index zip hash mismatch; re-downloading')
        UPLOAD.unlink()
if not UPLOAD.exists():
    try:
        download_index()
    except Exception as e:
        print('release download failed; will build from SlideVQA:', repr(e))
        UPLOAD.unlink(missing_ok=True)

def verify_index(path: Path) -> None:
    meta = path / 'meta.db'
    assert meta.exists(), f'missing {meta}'
    con = sqlite3.connect(meta)
    try:
        pages = con.execute('SELECT COUNT(*) FROM pages').fetchone()[0]
        decks = sorted({r[0] for r in con.execute('SELECT DISTINCT doc_id FROM pages')})
        words = con.execute('SELECT COUNT(*) FROM word_boxes').fetchone()[0]
    finally:
        con.close()
    print('index pages', pages, 'words', words, 'decks', len(decks))
    assert pages == EXPECTED_PAGES, f'expected {EXPECTED_PAGES} pages, got {pages}'
    assert set(decks) == set(DEV_DECKS), 'deck ids do not match the embedded gold'

if UPLOAD.exists():
    print('using', UPLOAD)
    tmp = Path('/content/phase0_index_upload')
    shutil.rmtree(tmp, ignore_errors=True)
    with zipfile.ZipFile(UPLOAD) as z:
        z.extractall(tmp)
    metas = list(tmp.rglob('meta.db'))
    assert metas, 'slidevqa_index.zip does not contain meta.db'
    src = metas[0].parent
    if INDEX.exists():
        shutil.rmtree(INDEX)
    shutil.move(str(src), str(INDEX))
else:
    if not (INDEX / 'manifest.json').exists():
        print('building the 8-deck dev index from SlideVQA ...')
        subprocess.run([
            sys.executable, 'scripts/prepare_slidevqa.py',
            '--out', 'data/slidevqa_phase0', '--split', 'train',
            '--decks-file', 'data/phase0_dev_decks.json',
            '--max-rows', '20000', '--n-decks', '9999', '--n-qa', '0',
            '--gold-out', 'data/slidevqa_phase0/gold_rebuilt.json',
        ], check=True)
        pdfs = sorted((Path('data/slidevqa_phase0') / 'raw').glob('*.pdf'))
        print('pdfs', len(pdfs))
        assert len(pdfs) == len(DEV_DECKS), f'expected {len(DEV_DECKS)} pdfs'
        subprocess.run([
            sys.executable, '-m', 'vvrag.ingest.run_ingest',
            '--input', 'data/slidevqa_phase0/raw',
            '--out', 'data/index_phase0',
            '--embedder', 'colqwen',
            '--model-id', 'vidore/colqwen2-v1.0',
            '--device', 'cuda', '--dtype', 'bfloat16',
            '--dpi', '150', '--upscale', '2.0',
            '--qdrant-path', 'data/index_phase0/qdrant',
            '--batch-size', '4',
        ], check=True)
        FRESH = True
    else:
        print('reusing existing data/index_phase0')

verify_index(INDEX)
print('FRESH', FRESH)


In [ ]:
import hashlib
import json
from pathlib import Path

import numpy as np

MODEL_ID = 'vidore/colqwen2-v1.0'
CACHE = Path('data/query_cache_phase0')
CACHE.mkdir(parents=True, exist_ok=True)

qs = json.loads(Path('data/phase0_dev_questions.json').read_text(encoding='utf-8'))
seen = {}
for q in qs:
    seen.setdefault(hashlib.sha1(q['question'].strip().encode('utf-8')).hexdigest(),
                    q['question'])
have = {p.stem for p in CACHE.glob('*.json')}
missing = {k: q for k, q in seen.items() if k not in have}
print('questions', len(seen), 'cached', len(have), 'missing', len(missing))

if missing:
    import time

    import torch
    from colpali_engine.models import ColQwen2, ColQwen2Processor

    processor = ColQwen2Processor.from_pretrained(MODEL_ID)
    model = ColQwen2.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.bfloat16,
        device_map='cuda',
    ).eval()

    t0 = time.time()
    dims = set()
    for i, (key, q) in enumerate(missing.items(), 1):
        batch = processor.process_queries([q]).to('cuda')
        with torch.no_grad():
            emb = model(**batch)
        emb = emb[0].detach().float().cpu().numpy()
        dims.add(tuple(emb.shape))
        (CACHE / f'{key}.json').write_text(json.dumps({
            'question': q,
            'vectors': emb.tolist(),
            'meta': {
                'model_id': MODEL_ID,
                'prompt': 'ColQwen2Processor.process_queries(q)',
                'source': 'phase0_dev_retrieval_colab.ipynb',
            },
        }), encoding='utf-8')
        if i % 10 == 0 or i == len(missing):
            print(f'{i}/{len(missing)}  {time.time()-t0:.1f}s')
    print('dims:', sorted(dims))

files = list(CACHE.glob('*.json'))
print('query vectors:', len(files))
assert seen.keys() <= {p.stem for p in files}, 'query cache is missing embedded questions'
for p in files[:3]:
    d = json.loads(p.read_text(encoding='utf-8'))
    print(p.name, np.asarray(d['vectors']).shape)
assert EXPECTED_QUESTIONS == len(seen)


In [ ]:
import os
from pathlib import Path

REPO = Path('/content/altVRAG')
if REPO.exists():
    os.chdir(REPO)
print('cwd', Path.cwd())


In [ ]:
SWEEP_SCRIPT = r'''"""Phase 0 retrieval sweep: keyword vs visual vs weighted RRF, with/without
the lexical reranker.

This is an offline, deterministic page-retrieval evaluator. It never calls a
reader, an examiner, or the network. Query vectors must already be cached
(`scripts/precompute_query_vectors.py` on a GPU box).

Examples
--------
    python scripts/tune_retrieval.py --index data/index --gold data/gold.json \
        --query-cache data/query_cache --allow-partial

    python scripts/tune_retrieval.py --index data/index --gold data/gold.json \
        --query-cache data/query_cache \
        --weights 1:1,1:2,1:3,1:5 --rrf-k 30,60,100 \
        --candidate-k 5,10,20 --rerank both

Outputs `rows.json` (meta + summary + per-question rows) and `summary.csv`
under `results/demo/retrieval_sweep_<timestamp>/` by default.
"""
from __future__ import annotations

import argparse
import csv
import datetime as dt
import json
import sys
from pathlib import Path

REPO = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(REPO / "src"))

from vvrag.eval.metrics import clopper_pearson_interval  # noqa: E402
from vvrag.ingest.index import connect_sqlite  # noqa: E402
from vvrag.retrieve.fusion import (rank_order, rrf_fuse,  # noqa: E402
                                   visual_page_scores)
from vvrag.retrieve.query_vectors import QueryVectorStore  # noqa: E402
from vvrag.retrieve.rerank import lexical_rerank  # noqa: E402
from vvrag.ui.qa import keyword_rank, page_words  # noqa: E402


def _load_items(path: str | Path) -> list[dict]:
    data = json.loads(Path(path).read_text(encoding="utf-8"))
    if isinstance(data, dict):
        data = data.get("items", [])
    return list(data)


def _acceptable(item: dict) -> list[tuple[str, int]]:
    got: list[tuple[str, int]] = []
    for ap in item.get("acceptable_pages") or []:
        if isinstance(ap, dict) and ap.get("doc_id") is not None:
            got.append((str(ap["doc_id"]), int(ap["page"])))
    if not got and item.get("doc_id") is not None:
        got.append((str(item["doc_id"]), int(item["page"])))
    seen, out = set(), []
    for key in got:
        if key not in seen:
            seen.add(key)
            out.append(key)
    return out


def _rank_of(order: list[tuple[str, int]],
             acceptable: list[tuple[str, int]]) -> int | None:
    best = None
    for key in acceptable:
        try:
            r = order.index(key) + 1
        except ValueError:
            continue
        if best is None or r < best:
            best = r
    return best


def _metrics(rows: list[dict], mode: str) -> dict:
    ranks = [r[mode]["rank"] for r in rows if r.get(mode)]
    n = len(ranks)
    if not n:
        return {"n": 0}
    h1 = sum(1 for r in ranks if r and r <= 1)
    h3 = sum(1 for r in ranks if r and r <= 3)
    h5 = sum(1 for r in ranks if r and r <= 5)
    found = [r for r in ranks if r]
    mrr = sum(1.0 / r for r in found) / n
    mrr_found = sum(1.0 / r for r in found) / len(found) if found else 0.0
    lo, hi = clopper_pearson_interval(h1, n)
    return {
        "n": n,
        "hit@1": round(h1 / n, 4),
        "hit@1_cp95_low": round(lo, 4),
        "hit@1_cp95_high": round(hi, 4),
        "hit@3": round(h3 / n, 4),
        "hit@5": round(h5 / n, 4),
        "mrr": round(mrr, 4),
        "mrr_found": round(mrr_found, 4),
        "found": len(found),
    }


def _parse_weights(raw: str) -> list[tuple[float, float]]:
    out = []
    for part in raw.split(","):
        part = part.strip()
        if not part:
            continue
        if ":" not in part:
            raise ValueError(f"weight must be keyword:visual, got {part!r}")
        k, v = part.split(":", 1)
        out.append((float(k), float(v)))
    return out or [(1.0, 1.0)]


def _parse_ints(raw: str) -> list[int]:
    return [int(x) for x in raw.split(",") if x.strip()]


def _candidate_list(order: list[tuple[str, int]], candidate_k: int,
                    words_map: dict) -> list[tuple[str, int, float, list]]:
    out = []
    for i, (d, p) in enumerate(order[:candidate_k]):
        out.append((d, p, 1.0 / (i + 1), words_map.get((d, p)) or []))
    return out


def _rerank(order: list[tuple[str, int]], question: str, candidate_k: int,
            words_map: dict) -> list[tuple[str, int]]:
    cand = _candidate_list(order, candidate_k, words_map)
    if len(cand) <= 1:
        return order
    ranked = lexical_rerank(question, cand, k=candidate_k)
    return [(d, p) for d, p, _s, _w in ranked] + [
        key for key in order[candidate_k:] if key not in {(d, p) for d, p, _s, _w in ranked}
    ]


def main() -> None:
    ap = argparse.ArgumentParser(description=__doc__)
    ap.add_argument("--index", default="data/index")
    ap.add_argument("--gold", default="data/gold.json")
    ap.add_argument("--query-cache", default="data/query_cache")
    ap.add_argument("--out", default="")
    ap.add_argument("--modes", default="keyword,visual,hybrid",
                    help="comma list: keyword,visual,hybrid")
    ap.add_argument("--weights", default="1:1,1:2,1:3,1:5",
                    help="hybrid keyword:visual weights, comma list")
    ap.add_argument("--rrf-k", default="30,60,100")
    ap.add_argument("--candidate-k", default="5,10,20")
    ap.add_argument("--rerank", choices=("off", "lexical", "both"), default="both")
    ap.add_argument("--limit", type=int, default=0, help="debug: first N items")
    ap.add_argument("--allow-partial", action="store_true",
                    help="evaluate only items with cached query vectors")
    args = ap.parse_args()

    index = Path(args.index)
    items = _load_items(args.gold)
    if args.limit:
        items = items[: args.limit]
    modes = [m.strip().lower() for m in args.modes.split(",") if m.strip()]
    modes = [m for m in modes if m in {"keyword", "visual", "hybrid"}]
    if not modes:
        raise SystemExit("no valid modes")
    weights = _parse_weights(args.weights)
    rrf_ks = _parse_ints(args.rrf_k)
    candidate_ks = _parse_ints(args.candidate_k)
    reranks = ["off", "lexical"] if args.rerank == "both" else [args.rerank]

    con = connect_sqlite(index / "meta.db")
    pages = [(str(r[0]), int(r[1])) for r in
             con.execute("SELECT doc_id, page FROM pages ORDER BY doc_id, page")]
    pages_set = set(pages)
    store = QueryVectorStore(args.query_cache)
    cached = {it["qid"]: store.get(it["question"]) is not None for it in items}
    missing = [it["qid"] for it in items if not cached[it["qid"]]]
    if missing and not args.allow_partial:
        print(f"ERROR: {len(missing)}/{len(items)} questions lack query vectors in "
              f"{args.query_cache}.", file=sys.stderr)
        print("Run scripts/precompute_query_vectors.py on a GPU box, or pass "
              "--allow-partial for a smaller diagnostic subset.", file=sys.stderr)
        raise SystemExit(2)
    if missing:
        keep = {it["qid"] for it in items if cached[it["qid"]]}
        items = [it for it in items if it["qid"] in keep]
        print(f"WARNING: partial cache; evaluating {len(items)} questions "
              f"({len(missing)} missing).")
    if not items:
        raise SystemExit("no questions with cached query vectors")

    # Build all source rankings once.
    base: dict[str, dict] = {}
    words_map: dict[tuple[str, int], list] = {}
    for i, it in enumerate(items, 1):
        q = it["question"]
        kw_full = keyword_rank(q, con, limit=None)
        kw = [(d, p) for d, p, _s, _w in kw_full]
        for d, p, _s, w in kw_full:
            words_map[(d, p)] = w
        vis = None
        hit = store.get(q)
        if hit is not None:
            vis_scores = visual_page_scores(index, pages, hit[0])
            vis = rank_order(vis_scores)
        # Ensure visual-only pages have words available for the reranker.
        if vis:
            for d, p in vis[:max(candidate_ks or [1])]:
                if (d, p) in pages_set and (d, p) not in words_map:
                    words_map[(d, p)] = page_words(con, d, p)
        base[it["qid"]] = {"item": it, "kw": kw, "vis": vis}
        print(f"[{i}/{len(items)}] {it['qid']}: keyword={len(kw)} "
              f"visual={'yes' if vis else 'no'}")

    configs: list[dict] = []
    if "keyword" in modes:
        for rk in reranks:
            configs.append({"mode": "keyword", "weights": None, "rrf_k": None,
                            "candidate_k": None, "rerank": rk})
    if "visual" in modes:
        for rk in reranks:
            # candidate_k only changes the lexical-reranker input; when the
            # reranker is off the full visual order is already used.
            for ck in (candidate_ks if rk != "off" else candidate_ks[:1]):
                configs.append({"mode": "visual", "weights": None, "rrf_k": None,
                                "candidate_k": ck, "rerank": rk})
    if "hybrid" in modes:
        for w in weights:
            for rk in rrf_ks:
                for rr in reranks:
                    for ck in (candidate_ks if rr != "off" else candidate_ks[:1]):
                        configs.append({"mode": "hybrid", "weights": w,
                                        "rrf_k": rk, "candidate_k": ck,
                                        "rerank": rr})

    rows: list[dict] = []
    for cfg in configs:
        label = _label(cfg)
        for it in items:
            b = base[it["qid"]]
            order = list(b["kw"])
            if cfg["mode"] == "visual":
                if b["vis"] is None:
                    continue
                order = list(b["vis"])
            elif cfg["mode"] == "hybrid":
                if b["vis"] is None:
                    continue
                fused = rrf_fuse([b["kw"], b["vis"]], k=int(cfg["rrf_k"]),
                                 weights=list(cfg["weights"]))
                order = [(d, p) for d, p, _s in fused]
            if cfg["rerank"] == "lexical":
                order = _rerank(order, it["question"], int(cfg["candidate_k"] or 5),
                                words_map)
            rank = _rank_of(order, _acceptable(it))
            rows.append({
                "config": label, "mode": cfg["mode"],
                "keyword_weight": cfg["weights"][0] if cfg["weights"] else None,
                "visual_weight": cfg["weights"][1] if cfg["weights"] else None,
                "rrf_k": cfg["rrf_k"], "candidate_k": cfg["candidate_k"],
                "rerank": cfg["rerank"], "qid": it["qid"],
                "question": it["question"], "rank": rank,
                "top10": [f"{d}/{p}" for d, p in order[:10]],
            })
        print(f"evaluated {label}: {sum(1 for r in rows if r['config'] == label)} rows")

    summary = []
    for cfg in configs:
        label = _label(cfg)
        sub = [r for r in rows if r["config"] == label]
        m = _metrics([{label: r} for r in sub], label)
        summary.append({**cfg, "label": label, **m})
    summary.sort(key=lambda s: (-float(s.get("hit@1", 0)),
                                -float(s.get("mrr", 0)),
                                -float(s.get("hit@3", 0))))

    stamp = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
    out = Path(args.out) if args.out else (
        REPO / "results" / "demo" / f"retrieval_sweep_{stamp}")
    out.mkdir(parents=True, exist_ok=True)
    meta = {
        "created_utc": dt.datetime.now(dt.timezone.utc).isoformat(),
        "index": str(index),
        "gold": str(args.gold),
        "query_cache": str(args.query_cache),
        "n_questions": len(items),
        "missing_query_vectors": len(missing),
        "modes": modes,
        "weights": weights,
        "rrf_k": rrf_ks,
        "candidate_k": candidate_ks,
        "rerank": reranks,
        "pages": len(pages),
    }
    (out / "rows.json").write_text(
        json.dumps({"meta": meta, "summary": summary, "rows": rows}, indent=2),
        encoding="utf-8")
    with (out / "summary.csv").open("w", newline="", encoding="utf-8") as f:
        fields = ["label", "mode", "keyword_weight", "visual_weight", "rrf_k",
                  "candidate_k", "rerank", "n", "hit@1", "hit@3", "hit@5",
                  "mrr", "mrr_found"]
        w = csv.DictWriter(f, fieldnames=fields, extrasaction="ignore")
        w.writeheader()
        w.writerows(summary)
    print("\n=== top configs ===")
    for s in summary[:15]:
        print(f"{s['label']:<55} hit@1={s.get('hit@1', 0):.3f} "
              f"hit@3={s.get('hit@3', 0):.3f} mrr={s.get('mrr', 0):.3f}")
    print(f"\nWROTE {out}")


def _label(cfg: dict) -> str:
    if cfg["mode"] == "keyword":
        return f"keyword|rerank={cfg['rerank']}"
    if cfg["mode"] == "visual":
        return f"visual|k={cfg['candidate_k']}|rerank={cfg['rerank']}"
    kw, vis = cfg["weights"]
    return (f"hybrid|w={kw:g}:{vis:g}|rrf={cfg['rrf_k']}|"
            f"k={cfg['candidate_k']}|rerank={cfg['rerank']}")


if __name__ == "__main__":
    main()
'''
from pathlib import Path
Path('scripts').mkdir(parents=True, exist_ok=True)
Path('scripts/tune_retrieval.py').write_text(SWEEP_SCRIPT, encoding='utf-8')
print('wrote scripts/tune_retrieval.py', len(SWEEP_SCRIPT), 'chars')


In [ ]:
import subprocess
import sys

cmd = [
    sys.executable, 'scripts/tune_retrieval.py',
    '--index', 'data/index_phase0',
    '--gold', 'data/phase0_dev_gold.json',
    '--query-cache', 'data/query_cache_phase0',
    '--out', 'results/demo/phase0_dev_sweep',
    '--modes', 'keyword,visual,hybrid',
    '--weights', '1:1,1:2,1:3,1:5',
    '--rrf-k', '30,60,100',
    '--candidate-k', '5,10,20',
    '--rerank', 'both',
]
print(' '.join(cmd))
subprocess.run(cmd, check=True)


In [ ]:
import json
from pathlib import Path

res = json.loads(Path('results/demo/phase0_dev_sweep/rows.json').read_text(encoding='utf-8'))
print('meta:', json.dumps(res['meta'], indent=1))
print('\nTop 20 configurations:')
for row in res['summary'][:20]:
    print(f"{row['label']:<58} hit@1={row['hit@1']:.3f} hit@3={row['hit@3']:.3f} "
          f"hit@5={row['hit@5']:.3f} mrr={row['mrr']:.3f} n={row['n']}")


In [ ]:
import json
import platform
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

import torch

prov = {
    'notebook': 'phase0_dev_retrieval_colab.ipynb',
    'repo_commit': subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip(),
    'python': sys.version,
    'platform': platform.platform(),
    'torch': torch.__version__,
    'cuda': torch.cuda.is_available(),
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    'dev_gold': len(DEV_GOLD),
    'dev_probes': len(DEV_PROBES),
    'decks': len(DEV_DECKS),
    'query_vectors': len(list(Path('data/query_cache_phase0').glob('*.json'))),
    'sweep': 'results/demo/phase0_dev_sweep/rows.json',
}
Path('results/phase0_provenance.json').write_text(json.dumps(prov, indent=1), encoding='utf-8')
print(json.dumps(prov, indent=1))

bundle = Path('/content/phase0_dev_retrieval_bundle.zip')
if bundle.exists():
    bundle.unlink()
include = [
    'results/demo/phase0_dev_sweep',
    'results/phase0_provenance.json',
    'data/phase0_dev_gold.json',
    'data/phase0_dev_probes.json',
    'data/phase0_dev_questions.json',
    'data/phase0_dev_decks.json',
    'data/query_cache_phase0',
]
with zipfile.ZipFile(bundle, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    for item in include:
        p = Path(item)
        if p.is_file():
            z.write(p, p.as_posix())
        elif p.is_dir():
            for f in sorted(p.rglob('*')):
                if f.is_file():
                    z.write(f, f.as_posix())
print('wrote', bundle, bundle.stat().st_size / 1e6, 'MB')


In [ ]:
from google.colab import files
files.download('/content/phase0_dev_retrieval_bundle.zip')


## What to send back

`phase0_dev_retrieval_bundle.zip` contains the sweep `rows.json`/`summary.csv`,
the 81 question vectors, the embedded dev gold/probes, and provenance. The
built index is intentionally **not** included (it is large); keep
`data/index_phase0` in Drive if you want to reuse it.

Report the top configurations by hit@1 / hit@3 / MRR, then compare the winner
on the **same 40 dev items** against the current keyword and equal-weight
hybrid baselines. Do not tune on frozen3 or frozen4.
